# **0. Importar librerias**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

In [2]:
os.listdir(data_folder)

NameError: name 'data_folder' is not defined

# **1. CARGA Y EXPLORACIÓN INICIAL**

In [ ]:
main_folder = Path.cwd().parent

data_folder = os.path.join(main_folder, 'data')


df, df_ip = [ pd.read_csv(os.path.join(data_folder, file)) for file in os.listdir(data_folder)]


In [ ]:
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [ ]:
df_ip.head()

,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


## 1.1 Revisión de columnas y tipo de datos

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         151112 non-null  int64  
 1   signup_time     151112 non-null  str    
 2   purchase_time   151112 non-null  str    
 3   purchase_value  151112 non-null  int64  
 4   device_id       151112 non-null  str    
 5   source          151112 non-null  str    
 6   browser         151112 non-null  str    
 7   sex             151112 non-null  str    
 8   age             151112 non-null  int64  
 9   ip_address      151112 non-null  float64
 10  class           151112 non-null  int64  
dtypes: float64(1), int64(4), str(6)
memory usage: 12.7 MB


### 💾 **Observaciones**


Existe un total de **11 columnas** y **151112 registros**.
    - 10 son variables independientes
    - 1 variable dependiente (`class`)

Existen 2 columnas que parecen estar relacionadas con fechas pero están como str: `signup_time`, `purchase_time`.

Hay 5 columnas **numéricas** (`user_id`, `purchase_value`, `user_id`, `age`,`ip_address` y `class`), las 6 restantes son **str**.


No existen datos nulos aparentes.





In [ ]:
df_ip.info()

<class 'pandas.DataFrame'>
RangeIndex: 138846 entries, 0 to 138845
Data columns (total 3 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   lower_bound_ip_address  138846 non-null  float64
 1   upper_bound_ip_address  138846 non-null  int64  
 2   country                 138846 non-null  str    
dtypes: float64(1), int64(1), str(1)
memory usage: 3.2 MB


### 💾 **Observaciones**


Existe un total de 3 columnas y 138846.

- 2 son numéricas relacionadas con limites de ip
- 1 de estas es str relacionada con el país de donde se efectuo la compra.



📢 **Conclusión**

Según lo señalado se debe revisar las columnas para ver si es necesario transformar alguna columna. En cuanto a las features relacionadas con tiempo deben ser cambiadas a su tipo correcto.

Revisar si es que en realidad no hay nulos como datos mal inputados, categorias faltantes u otros errores también debe estar presente en el análisis.

**Columnas a corregir**
- signup_time y purchase_time son object --> convertir a datetime
- source, browser, sex son object --> convertir a category (reducir memoria)


## 1.2 Revisión numérica somera

In [ ]:
# No ver notación cientifica
pd.set_option('display.float_format', '{:.2f}'.format)

df.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
user_id,151112.00,200171.04,115369.29,2.00,100642.50,199958.00,300054.00,400000.00
purchase_value,151112.00,36.94,18.32,9.00,22.00,35.00,49.00,154.00
age,151112.00,33.14,8.62,18.00,27.00,33.00,39.00,76.00
ip_address,151112.00,2152145330.96,1248497030.10,52093.50,1085933882.53,2154770162.41,3243257679.72,4294850499.68
class,151112.00,0.09,0.29,0.00,0.00,0.00,0.00,1.00


In [ ]:
df_ip.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
lower_bound_ip_address,138846,2724531563,897521520,16777216,1919930368,3230887296,3350465280,3758096128
upper_bound_ip_address,138846,2724557062,897497915,16777471,1920008191,3230887551,3350465919,3758096383


### 💾 **Observaciones**

`purchase_value`: 
- **promedio**: 36.94 $
- **std**: 18.32 $
- **min**: 9 $
- **max**: 400000.00 $

`age`:

- **promedio**: 33 años
- **std**: 8.62 años
- **min**: 18 años
- **max**: 76 años



`lower_bound_ip_address`:

- **min**: 16777216
- **max**: 3758096128


`upper_bound_ip_address`:

- **min**: 16777471
- **max**: 3758096383

📢 **Conclusión**

El 75 % de los datos esta por debajo de los 49 $ de `purchase_value`. Considerando esto y el valor del maximo es interesante ver cuantos datos están fuera del 75 % y que tan altos son.

  - El std es bastante alto lo que debe ser principalmente por los valores ya nombrados.

La edad promedio de los clientes es de  33 años.
  - La edad mínima es 18 en sintonia con el negocio y la máxima de 76.
  - El std es alto pero no desmezutado como el de la columna anterior


En cuanto IP, podremos ver algo mas relacionandolo con el país.


`df` y `df_ip` se relacionan entre las ip. Con el ip de `df` hay que encontrar dentro de que rango cae y se le puede asignar un país a esa ip.

##  1.4. Unir los dataframes

In [ ]:
df_ordenado = df.sort_values('ip_address')
df_ip_ordenado = df_ip.sort_values('lower_bound_ip_address')

df_merged = pd.merge_asof(
    df_ordenado,
    df_ip_ordenado,
    left_on='ip_address',
    right_on='lower_bound_ip_address',
    direction='backward'
)

In [ ]:
df_merged.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,lower_bound_ip_address,upper_bound_ip_address,country
0,62421,2015-02-16 00:17:05,2015-03-08 10:00:39,46,ZCLZTAJPCRAQX,Direct,Safari,M,36,52093.50,0,NaN,NaN,NaN
1,173212,2015-03-08 04:03:22,2015-03-20 17:23:45,33,YFGYOALADBHLT,Ads,IE,F,30,93447.14,0,NaN,NaN,NaN
2,242286,2015-05-17 16:45:54,2015-05-26 08:54:34,33,QZNVQTUITFTHH,Direct,FireFox,F,32,105818.50,0,NaN,NaN,NaN
3,370003,2015-03-03 19:58:39,2015-05-28 21:09:13,33,PIBUQMBIELMMG,Ads,IE,M,40,117566.66,0,NaN,NaN,NaN
4,119824,2015-03-20 00:31:27,2015-04-05 07:31:46,55,WFIIFCPIOGMHT,Ads,Safari,M,38,131423.79,0,NaN,NaN,NaN


**Detalle**

Uso merge_assoft porque me permite hacer un merge utilizando rangos.
La idea es ordenar ambos dataset por la columna donde quiera comparar los rangos.

Utilizo `ip_address` en la tabla izquierda y `lower_bound_ip_address` en la derecha
para buscar el valor más cercano o igual hacia atrás de cada IP.

Aun así, esto dejará casos donde `ip_address` > `upper_bound_ip_address`.
En esos casos el país se marca como 'Unknown' ya que la IP no pertenece a ese rango.


In [ ]:
df_merged.isnull().sum()

user_id                     0
signup_time                 0
purchase_time               0
purchase_value              0
device_id                   0
source                      0
browser                     0
sex                         0
age                         0
ip_address                  0
class                       0
lower_bound_ip_address    634
upper_bound_ip_address    634
country                   634
dtype: int64